In [1]:
print("OpinionAI - DistilBERT Model Training")

OpinionAI - DistilBERT Model Training


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
!pip install -q transformers datasets accelerate scikit-learn

In [4]:
!git clone https://github.com/rohanpython9229-collab/Opinion_metrix_AI_Sentiment_Transformer.git

Cloning into 'Opinion_metrix_AI_Sentiment_Transformer'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 23 (delta 9), reused 16 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 42.14 KiB | 463.00 KiB/s, done.
Resolving deltas: 100% (9/9), done.


In [5]:
!cd /content/Opinion_metrix_AI_Sentiment_Transformer && git pull origin main

From https://github.com/rohanpython9229-collab/Opinion_metrix_AI_Sentiment_Transformer
 * branch            main       -> FETCH_HEAD
Already up to date.


In [6]:
import pandas as pd

df = pd.read_csv(
    "/content/Opinion_metrix_AI_Sentiment_Transformer/data/clean_reviews.csv"
)

df.head()

,review,sentiment
0,"This desk lamp has both good and bad points, o...",Neutral
1,"The smartwatch exceeded my expectations, aweso...",Positive
2,"Really top-notch bluetooth speaker, exactly as...",Positive
3,Shipping took a while but the laptop backpack ...,Positive
4,"FAST SHIPPING, HOWEVER THE SMARTWATCH ITSELF I...",Negative


In [7]:
df.shape

(1453, 2)

In [8]:
df.columns

Index(['review', 'sentiment'], dtype='object')

In [9]:
label2id = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

id2label = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

In [10]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["sentiment"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Train: 1162
Validation: 145
Test: 146


In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [12]:
print(df.columns)
print(df.shape)

Index(['review', 'sentiment'], dtype='object')
(1453, 2)


In [13]:
y_train = y_train.map(label2id)
y_val = y_val.map(label2id)
y_test = y_test.map(label2id)

print(y_train.head())

445     1
1017    0
484     0
1101    2
1347    2
Name: sentiment, dtype: int64


In [14]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_dict({
    "text": X_train.tolist(),
    "label": y_train.tolist()
})

val_dataset = Dataset.from_dict({
    "text": X_val.tolist(),
    "label": y_val.tolist()
})

test_dataset = Dataset.from_dict({
    "text": X_test.tolist(),
    "label": y_test.tolist()
})

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1162
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 145
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 146
    })
})

In [15]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [16]:
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/1162 [00:00<?, ? examples/s]

Map:   0%|          | 0/145 [00:00<?, ? examples/s]

Map:   0%|          | 0/146 [00:00<?, ? examples/s]

In [17]:
tokenized_dataset["train"].features

{'text': Value('string'),
 'label': Value('int64'),
 'input_ids': List(Value('int32')),
 'token_type_ids': List(Value('int8')),
 'attention_mask': List(Value('int8'))}

In [18]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert-sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True,
    logging_steps=20,
    report_to="none"
)

In [20]:
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions)
    }

In [21]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics
)

In [22]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.413866,0.127192,0.986207
2,0.247025,0.087690,0.986207
3,0.251027,0.087292,0.986207


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
test_results = trainer.evaluate(tokenized_dataset["test"])

print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy
0.251027,0.173779,3,0.972603


{'eval_loss': 0.1737794280052185, 'eval_accuracy': 0.9726027397260274}


In [24]:
from sklearn.metrics import classification_report, confusion_matrix

predictions = trainer.predict(tokenized_dataset["test"])

y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print(classification_report(
    y_true,
    y_pred,
    target_names=["Negative", "Neutral", "Positive"]
))

              precision    recall  f1-score   support

    Negative       0.92      1.00      0.96        48
     Neutral       1.00      0.96      0.98        49
    Positive       1.00      0.96      0.98        49

    accuracy                           0.97       146
   macro avg       0.97      0.97      0.97       146
weighted avg       0.97      0.97      0.97       146



In [25]:
cm = confusion_matrix(y_true, y_pred)

print(cm)

[[48  0  0]
 [ 2 47  0]
 [ 2  0 47]]


In [26]:
model_dir = "./OpinionAI-Sentiment-DistilBERT"

model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [28]:
from huggingface_hub import login

login()

In [29]:
repo_id = "rohanpython9229/OpinionAI-Sentiment-DistilBERT"

model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...vdvwiqo/model.safetensors:   0%|          |  573kB /  268MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/rohanpython9229/OpinionAI-Sentiment-DistilBERT/commit/7288a4ab079043abb79b9988ad0400fbbb1c4c0f', commit_message='Upload tokenizer', commit_description='', oid='7288a4ab079043abb79b9988ad0400fbbb1c4c0f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/rohanpython9229/OpinionAI-Sentiment-DistilBERT', endpoint='https://huggingface.co', repo_type='model', repo_id='rohanpython9229/OpinionAI-Sentiment-DistilBERT'), pr_revision=None, pr_num=None)